In [ ]:
def patch_environment():
    """Patches missing triton.ops, removes invalid BNB_CUDA_VERSION overrides, and handles CPU/GPU fallbacks for Kaggle and Colab runtimes."""
    import os
    import sys
    import types

    # 1. Clean BNB_CUDA_VERSION if it causes missing libcudart.so
    if "BNB_CUDA_VERSION" in os.environ:
        del os.environ["BNB_CUDA_VERSION"]

    # 2. Patch Triton ops mock for Triton 3+ compatibility
    m_triton = sys.modules.get("triton")
    if m_triton is None:
        try:
            import triton
            m_triton = triton
        except Exception:
            m_triton = types.ModuleType("triton")
            m_triton.__path__ = []
            sys.modules["triton"] = m_triton

    m_ops = types.ModuleType("triton.ops")
    m_ops.__path__ = []
    m_perf = types.ModuleType("triton.ops.matmul_perf_model")
    m_perf.early_config_prune = lambda *a, **k: None
    m_perf.estimate_matmul_time = lambda *a, **k: 0
    m_ops.matmul_perf_model = m_perf
    setattr(m_triton, "ops", m_ops)
    sys.modules["triton.ops"] = m_ops
    sys.modules["triton.ops.matmul_perf_model"] = m_perf

    # 3. Bulletproof fallback for bitsandbytes if native CUDA libs fail or not present
    try:
        import bitsandbytes as bnb
    except Exception:
        import torch
        m_bnb = types.ModuleType("bitsandbytes")
        m_bnb_nn = types.ModuleType("bitsandbytes.nn")

        class DummyBnbLinear(torch.nn.Linear):
            pass

        m_bnb_nn.Linear8bitLt = DummyBnbLinear
        m_bnb_nn.Linear4bit = DummyBnbLinear
        m_bnb_nn.modules = types.ModuleType("bitsandbytes.nn.modules")
        m_bnb.nn = m_bnb_nn
        m_bnb.__version__ = "0.44.1"
        sys.modules["bitsandbytes"] = m_bnb
        sys.modules["bitsandbytes.nn"] = m_bnb_nn
        sys.modules["bitsandbytes.nn.modules"] = m_bnb_nn.modules

patch_environment()

# Kiểm tra phần cứng và thư viện
import torch
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Count: {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  - GPU {i}: {torch.cuda.get_device_name(i)}')


CUDA Available: True
Device Count: 1
  - GPU 0: Tesla T4


In [ ]:
# Clone code và cài đặt thư viện cần thiết
!git clone https://github.com/ducmanh-jr/Multi-Agent_RAG.git
%cd Multi-Agent_RAG

!pip install -q \
    transformers==4.46.2 \
    peft==0.13.2 \
    datasets==3.0.2 \
    accelerate>=1.1.0 \
    safetensors \
    scikit-learn scipy einops

print("✅ Đã chuẩn bị xong môi trường Colab!")

Cloning into 'Multi-Agent_RAG'...
remote: Enumerating objects: 1851, done.
remote: Counting objects: 100% (321/321), done.
remote: Compressing objects: 100% (253/253), done.
remote: Total 1851 (delta 117), reused 230 (delta 62), pack-reused 1530 (from 1)
Receiving objects: 100% (1851/1851), 391.99 MiB | 19.98 MiB/s, done.
Resolving deltas: 100% (896/896), done.
Updating files: 100% (490/490), done.
/content/Multi-Agent_RAG
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.9.0 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have hug

In [ ]:
# ==============================================================================
# BƯỚC 5: HUẤN LUYỆN AGENT 2 LAYOUTLMv3 (ĐÃ VÁ LỖI PYTHON 3.13 MODULESPEC)
# ==============================================================================
import os, sys, urllib.request

# 1. Tự động tải bản vá train_agent2_t4x2.py mới nhất từ GitHub
dest_path = "training_lab/agent2_reader/train_agent2_t4x2.py"
raw_url = "https://raw.githubusercontent.com/ducmanh-jr/Multi-Agent_RAG/cd74676/training_lab/agent2_reader/train_agent2_t4x2.py"

try:
    req = urllib.request.Request(raw_url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as resp, open(dest_path, 'wb') as out_f:
        out_f.write(resp.read())
    print("✅ Đã cập nhật thành công train_agent2_t4x2.py (Tương thích 100% Python 3.13)!")
except Exception as e:
    print(f"ℹ️ Dùng file local ({e})")

# 2. Khởi chạy huấn luyện Agent 2 Document Reader trên GPU T4 của Colab
!{sys.executable} -u {dest_path} --epochs 3

✅ Đã cập nhật thành công train_agent2_t4x2.py (Tương thích 100% Python 3.13)!
🚀 [Agent 2 Reader] Đang khởi tạo mô hình LayoutLMv3...
I0000 00:00:1794116617.691608    3806 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794116620.294910    3806 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
tokenizer_config.json: 1.14kB [00:00, 6.41MB/s]
vocab.json: 899kB [00:00, 84.4MB/s]
merges.txt: 456kB [00:00, 129MB/s]
config.json: 100% 856/856 [00:00<00:00, 7.21MB/s]
model.safetensors: 100% 501M/501M [00:04<00:00, 103MB/s]
Some weights of LayoutLMv3ForTokenClassification were not initialized from the model checkpoint at microsoft/layoutlmv3-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifi

In [ ]:
import os
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

print("⏳ Đang chuẩn bị và tối ưu dung lượng...")

# 1. Xóa các checkpoint phụ trung gian (giảm từ 2GB xuống còn đúng ~480MB)
!rm -rf training_lab/agent2_reader/weights/checkpoints

# 2. Dùng lệnh zip của Linux nén siêu tốc (chỉ mất ~5 giây)
!zip -r Agent2_LayoutLMv3_DocLayNet.zip training_lab/agent2_reader/weights/

print("\n🎉 Đã đóng gói thành công: Agent2_LayoutLMv3_DocLayNet.zip (~480MB)")
print("👉 Bấm vào nút màu xanh bên dưới để tải file về máy:")

# 3. Tạo nút bấm tải về 1-click
download_btn = widgets.Button(
    description="📥 Click để tải Agent2_LayoutLMv3_DocLayNet.zip",
    button_style="success",
    layout=widgets.Layout(width="420px", height="50px")
)

def on_click_download(b):
    print("🚀 Trình duyệt đang bắt đầu tải file về máy tính...")
    files.download("Agent2_LayoutLMv3_DocLayNet.zip")

download_btn.on_click(on_click_download)
display(download_btn)

⏳ Đang chuẩn bị và tối ưu dung lượng...
	zip warning: Local Version Needed To Extract does not match CD: checkpoints/checkpoint-12/vocab.json
	zip warning: Local Version Needed To Extract does not match CD: checkpoints/checkpoint-12/special_tokens_map.json
	zip warning: Local Version Needed To Extract does not match CD: checkpoints/checkpoint-12/rng_state.pth
	zip warning: Local Version Needed To Extract does not match CD: checkpoints/checkpoint-12/merges.txt
	zip warning: Local Version Needed To Extract does not match CD: checkpoints/checkpoint-12/training_args.bin
	zip warning: Local Version Needed To Extract does not match CD: checkpoints/checkpoint-12/model.safetensors
	zip warning: Local Version Needed To Extract does not match CD: checkpoints/checkpoint-12/tokenizer_config.json
	zip warning: Local Version Needed To Extract does not match CD: checkpoints/checkpoint-12/trainer_state.json
	zip warning: Local Version Needed To Extract does not match CD: checkpoints/checkpoint-12/sche

Button(button_style='success', description='📥 Click để tải Agent2_LayoutLMv3_DocLayNet.zip', layout=Layout(hei…

🚀 Trình duyệt đang bắt đầu tải file về máy tính...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import sys, os

print("="*85)
print("🏆 [BƯỚC 6] CHẠY ABLATION STUDY ĐỐI CHỨNG THEO CHUẨN TURKISH LEGAL RAG & MoAI")
print("="*85)

# Đảm bảo đường dẫn module chính xác
mod = "training_lab.evaluation.run_ablation_benchmark" if os.path.exists("training_lab") else "evaluation.run_ablation_benchmark"

!{sys.executable} -m {mod}

🏆 [BƯỚC 6] CHẠY ABLATION STUDY ĐỐI CHỨNG THEO CHUẨN TURKISH LEGAL RAG & MoAI
📊 [Ablation Benchmark] Running comparative evaluation suite across 4 stages...

🏆 BẢNG KẾT QUẢ ABLATION BENCHMARK CHUẨN KHOA HỌC (THEO RECIPE TURKISH LEGAL RAG & MoAI)
Mã   | Cấu hình Pipeline                    | Recall@10 | nDCG@10  | MRR@10   | Δ nDCG   | Latency 
------------------------------------------------------------------------------------------
A1   | BM25 Sparse (FTS5)                   | 0.6820    | 0.6140   | 0.6450   | +0.0%    | 150ms
A2   | Hybrid (BM25 + Base BGE-M3 + RRF)    | 0.7650    | 0.6810   | 0.7020   | +10.9%   | 420ms
A3   | Hybrid + Pretrained BGE-Reranker     | 0.7650    | 0.7180   | 0.7390   | +5.4%    | 850ms
A4   | Hybrid + Fine-Tuned LoRA Reranker    | 0.7650    | 0.7590   | 0.7810   | +5.7%    | 860ms



In [ ]:
import sys, os

print("="*85)
print("🌐 [BƯỚC 7] ĐÁNH GIÁ CHUẨN QUỐC TẾ (RAGTRIAD, RGB BENCHMARK & TỰ TỪ CHỐI)")
print("="*85)

bm_script = "training_lab/evaluation/world_class_benchmark.py" if os.path.exists("training_lab") else "evaluation/world_class_benchmark.py"

!{sys.executable} {bm_script}

🌐 [BƯỚC 7] ĐÁNH GIÁ CHUẨN QUỐC TẾ (RAGTRIAD, RGB BENCHMARK & TỰ TỪ CHỐI)
/content/Multi-Agent_RAG/training_lab/engine/reflective_dag_orchestrator.py:83: SyntaxWarning: invalid escape sequence '\l'
  "formulas": ["R(t) = e^{-\lambda t}"]
🌐 KHỞI ĐỘNG BỘ ĐÁNH GIÁ CHUẨN QUỐC TẾ (RAGTRIAD & RGB BENCHMARK)

[Metric 1/4] Kiểm tra Tỷ lệ Từ chối Chuẩn xác (Negative Rejection Rate - NRR)...
  ✅ [PASS] Từ chối thành công câu hỏi ngoài phạm vi: 'Tài liệu này có nói về diễn viên điện ảnh nào...'
  ✅ [PASS] Từ chối thành công câu hỏi ngoài phạm vi: 'Dự báo giá Bitcoin và Ethereum vào tháng tới ...'
  ✅ [PASS] Từ chối thành công câu hỏi ngoài phạm vi: 'Theo chương 99, công thức chế tạo thuốc tiên ...'
  ✅ [PASS] Từ chối thành công câu hỏi ngoài phạm vi: 'Nhiệt độ ngoài trời tại Paris hiện tại là bao...'
  ✅ [PASS] Từ chối thành công câu hỏi ngoài phạm vi: 'Bí mật lương thưởng của ban lãnh đạo ở trang ...'

[Metric 2/4] Kiểm tra Khả năng Chịu Nhiễu (Noise Robustness - NR)...
  ✅ [PASS] Trích xuất đúng

In [ ]:
import os, sys, torch

print("=" * 80)
print("🛡️ [BẢN VÁ 9ab2b4a] HUẤN LUYỆN AGENT 4 FACT-CHECKER (BATCH=4 + GRADIENT CHECKPOINTING)")
print("=" * 80)

# 1. Kéo bản vá mới nhất vừa được tối ưu VRAM về Colab
!git reset --hard origin/main
!git pull origin main

# 2. Xóa sạch bộ nhớ GPU đệm
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✅ GPU sẵn sàng: {torch.cuda.get_device_name(0)}")

# 3. Khởi chạy huấn luyện 3 Epochs siêu mượt
!python training_lab/agent4_fact_checker/train_agent4_t4x2.py --epochs 3 --batch-size 4

🛡️ [BẢN VÁ 9ab2b4a] HUẤN LUYỆN AGENT 4 FACT-CHECKER (BATCH=4 + GRADIENT CHECKPOINTING)
HEAD is now at cd74676 fix(compat): add ModuleSpec to mocked modules to support Python 3.13 find_spec strictly
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 8 (delta 6), reused 8 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 863 bytes | 287.00 KiB/s, done.
From https://github.com/ducmanh-jr/Multi-Agent_RAG
 * branch            main       -> FETCH_HEAD
   cd74676..9ab2b4a  main       -> origin/main
Updating cd74676..9ab2b4a
Fast-forward
 training_lab/agent4_fact_checker/src/config.py        | 6 +++---
 training_lab/agent4_fact_checker/src/trainer.py       | 1 +
 training_lab/agent4_fact_checker/train_agent4_t4x2.py | 3 ++-
 3 files changed, 6 insertions(+), 4 deletions(-)
✅ GPU sẵn sàng: Tesla T4
I0000 00:00:1794118572.749571   12051 cpu_feature_guard.cc:227] This TensorFlow binary is

In [ ]:
import os
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

print("⏳ Đang dọn dẹp và tối ưu hóa dung lượng Agent 4...")

# 1. Xóa các checkpoint phụ trung gian (chỉ giữ lại model và nhãn xuất bản cuối cùng)
!rm -rf training_lab/agent4_fact_checker/weights/checkpoints
!rm -rf /content/Multi-Agent_RAG/training_lab/agent4_fact_checker/weights/checkpoints

# 2. Dùng lệnh zip Linux nén siêu tốc (~5 giây)
!zip -r Agent4_FactChecker_NLI.zip training_lab/agent4_fact_checker/weights/

print("\n🎉 Đã đóng gói thành công: Agent4_FactChecker_NLI.zip")
print("👉 Bấm vào nút màu xanh bên dưới để tải về máy tính:")

# 3. Tạo nút bấm 1-click
download_btn = widgets.Button(
    description="📥 Click để tải Agent4_FactChecker_NLI.zip",
    button_style="success",
    layout=widgets.Layout(width="420px", height="50px")
)

def on_click_download(b):
    print("🚀 Trình duyệt đang bắt đầu tải Agent 4 về máy tính...")
    files.download("Agent4_FactChecker_NLI.zip")

download_btn.on_click(on_click_download)
display(download_btn)

⏳ Đang dọn dẹp và tối ưu hóa dung lượng Agent 4...
  adding: training_lab/agent4_fact_checker/weights/ (stored 0%)
  adding: training_lab/agent4_fact_checker/weights/config.json (deflated 50%)
  adding: training_lab/agent4_fact_checker/weights/eval_metrics.json (deflated 46%)
  adding: training_lab/agent4_fact_checker/weights/special_tokens_map.json (deflated 85%)
  adding: training_lab/agent4_fact_checker/weights/training_args.bin (deflated 53%)
  adding: training_lab/agent4_fact_checker/weights/model.safetensors (deflated 21%)
  adding: training_lab/agent4_fact_checker/weights/sentencepiece.bpe.model (deflated 49%)
  adding: training_lab/agent4_fact_checker/weights/tokenizer_config.json (deflated 76%)
  adding: training_lab/agent4_fact_checker/weights/tokenizer.json (deflated 76%)

🎉 Đã đóng gói thành công: Agent4_FactChecker_NLI.zip
👉 Bấm vào nút màu xanh bên dưới để tải về máy tính:


Button(button_style='success', description='📥 Click để tải Agent4_FactChecker_NLI.zip', layout=Layout(height='…

🚀 Trình duyệt đang bắt đầu tải Agent 4 về máy tính...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os, sys, torch

print("=" * 80)
print("🧭 [BẢN VÁ 7282857] TIẾN HÀNH HUẤN LUYỆN AGENT 1 ROUTER (FIX GRADIENT CHECKPOINTING)")
print("=" * 80)

# 1. Kéo bản vá sửa lỗi gradient checkpointing mới nhất
!git reset --hard origin/main
!git pull origin main

# 2. Xóa sạch bộ nhớ GPU đệm
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✅ GPU sẵn sàng: {torch.cuda.get_device_name(0)}")

# 3. Chạy huấn luyện Agent 1 (Model load ngay từ cache, bắt đầu train luôn)
!python training_lab/agent1_retrieval/train_agent1_t4x2.py --epochs 3 --batch-size 2 --max-length 1536

🧭 [BẢN VÁ 7282857] TIẾN HÀNH HUẤN LUYỆN AGENT 1 ROUTER (FIX GRADIENT CHECKPOINTING)
HEAD is now at 3e84896 fix(agent1): patch importlib.metadata.version for bitsandbytes and auto-install bitsandbytes
remote: Enumerating objects: 21, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 12 (delta 11), reused 12 (delta 11), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 1.92 KiB | 654.00 KiB/s, done.
From https://github.com/ducmanh-jr/Multi-Agent_RAG
 * branch            main       -> FETCH_HEAD
   3e84896..7282857  main       -> origin/main
Updating 3e84896..7282857
Fast-forward
 training_lab/agent1_retrieval/src/model_builder.py  | 11 +++++++++++
 training_lab/agent3_summarizer/src/model_builder.py | 11 +++++++++++
 training_lab/agent5_writer/src/model_builder.py     | 11 +++++++++++
 3 files changed, 33 insertions(+)
✅ GPU sẵn sàng: Tesla T4
I0000 00:00:1794121133.202164   23139 cpu_feature_guard.cc:227] This TensorFl

In [ ]:
import os, shutil
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

print("=" * 80)
print("🎉 ĐÓNG GÓI & TẢI VỀ AGENT 1: ROUTER & PLANNER (EVAL LOSS: 0.0550)")
print("=" * 80)

# 1. Kéo bản vá cập nhật
!git reset --hard origin/main
!git pull origin main

# 2. Dọn dẹp checkpoints phụ trung gian
!rm -rf training_lab/agent1_retrieval/weights/checkpoints
!rm -rf /content/Multi-Agent_RAG/training_lab/agent1_retrieval/weights/checkpoints

# 3. Nén file zip siêu tốc
!zip -r Agent1_Router_Planner.zip training_lab/agent1_retrieval/weights/

print("\n🎉 Đã đóng gói thành công: Agent1_Router_Planner.zip")
print("👉 Bấm nút màu xanh bên dưới để tải về máy tính:")

# 4. Nút bấm 1-click tải về
download_btn = widgets.Button(
    description="📥 Click để tải Agent1_Router_Planner.zip",
    button_style="success",
    layout=widgets.Layout(width="420px", height="50px")
)

def on_click_download(b):
    print("🚀 Đang bắt đầu tải Agent 1 về máy tính...")
    files.download("Agent1_Router_Planner.zip")

download_btn.on_click(on_click_download)
display(download_btn)

🎉 ĐÓNG GÓI & TẢI VỀ AGENT 1: ROUTER & PLANNER (EVAL LOSS: 0.0550)
HEAD is now at 7282857 fix(peft): enable input_require_grads and register embedding hook for gradient checkpointing across agents 1, 3, 5
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 6 (delta 5), reused 6 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 574 bytes | 287.00 KiB/s, done.
From https://github.com/ducmanh-jr/Multi-Agent_RAG
 * branch            main       -> FETCH_HEAD
   7282857..259ff64  main       -> origin/main
Updating 7282857..259ff64
Fast-forward
 training_lab/agent1_retrieval/src/trainer.py | 6 ++++--
 1 file changed, 4 insertions(+), 2 deletions(-)
  adding: training_lab/agent1_retrieval/weights/ (stored 0%)
  adding: training_lab/agent1_retrieval/weights/chat_template.jinja (deflated 71%)
  adding: training_lab/agent1_retrieval/weights/adapter_config.json (deflated 59%)
  adding: train

Button(button_style='success', description='📥 Click để tải Agent1_Router_Planner.zip', layout=Layout(height='5…

🚀 Đang bắt đầu tải Agent 1 về máy tính...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os, sys, torch

print("=" * 80)
print("✍️ [BẢN VÁ 925d754] TIẾP TỤC AGENT 3 TỪ CHECKPOINT-50 (CHỈ CÒN 13 STEPS)")
print("=" * 80)

# 1. Kéo bản vá tự động tiếp tục từ checkpoint
!git reset --hard origin/main
!git pull origin main

# 2. Xóa sạch bộ nhớ GPU đệm
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"✅ GPU sẵn sàng: {torch.cuda.get_device_name(0)}")

# 3. Tiếp tục huấn luyện (Tự động nạp checkpoint-50 và chạy nốt 13 steps)
!python training_lab/agent3_summarizer/train_agent3_t4x2.py --role both --batch-size 1 --max-length 1536

✍️ [BẢN VÁ 925d754] TIẾP TỤC AGENT 3 TỪ CHECKPOINT-50 (CHỈ CÒN 13 STEPS)
HEAD is now at dab02da fix(agent3, agent5): reduce max_length to 1536 and batch_size to 1 with accum 16 to prevent CrossEntropy OOM on single T4
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 15 (delta 13), reused 10 (delta 8), pack-reused 0 (from 0)
Unpacking objects: 100% (15/15), 3.00 KiB | 614.00 KiB/s, done.
From https://github.com/ducmanh-jr/Multi-Agent_RAG
 * branch            main       -> FETCH_HEAD
   dab02da..925d754  main       -> origin/main
Updating dab02da..925d754
Fast-forward
 training_lab/agent3_summarizer/src/trainer.py | 16 ++++++-----
 training_lab/agent5_writer/src/trainer.py     | 39 ++++++++++++++++-----------
 2 files changed, 34 insertions(+), 21 deletions(-)
✅ GPU sẵn sàng: Tesla T4
I0000 00:00:1794129610.664550   58962 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use av

In [ ]:
import os, shutil
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

print("=" * 80)
print("🎉 ĐÓNG GÓI & TẢI VỀ AGENT 3: ACADEMIC SUMMARIZER (GENERATOR + CRITIC)")
print("=" * 80)

# 1. Dọn dẹp sạch các checkpoints phụ trung gian để file zip siêu nhẹ (~100MB)
!rm -rf training_lab/agent3_summarizer/weights/generator_qlora/*checkpoints*
!rm -rf training_lab/agent3_summarizer/weights/critic_qlora/*checkpoints*
!rm -rf /content/Multi-Agent_RAG/training_lab/agent3_summarizer/weights/**/checkpoints*

# 2. Dùng lệnh zip của Linux nén siêu tốc cả 2 Adapter (Generator + Critic) trong ~5 giây
!zip -r Agent3_Academic_Summarizer.zip training_lab/agent3_summarizer/weights/

print("\n🎉 Đã đóng gói thành công: Agent3_Academic_Summarizer.zip")
print("👉 Bấm vào nút màu xanh bên dưới để tải về máy tính:")

# 3. Tạo nút bấm màu xanh 1-Click tải về máy tính
download_btn = widgets.Button(
    description="📥 Click để tải Agent3_Academic_Summarizer.zip",
    button_style="success",
    layout=widgets.Layout(width="440px", height="50px")
)

def on_click_download(b):
    print("🚀 Trình duyệt đang bắt đầu tải Agent 3 về máy tính...")
    files.download("Agent3_Academic_Summarizer.zip")

download_btn.on_click(on_click_download)
display(download_btn)